
# Genetic Algorithm — Two Generations & Convergence Check

This notebook follows the **exact same manual pattern used in the worked notes** — Initial
Population → Fitness → Selection Probability → Expected Count → Actual Count → Mating Pool →
Crossover → Mutation — but runs it **twice** (Population 1 → Population 2) for each problem, then
performs an explicit **convergence check** comparing the two generations.

**Convergence is assessed using three signals, compared between Population 1 and Population 2:**

| Signal | What it means |
|---|---|
| **Best value** | Did the best (max, for Q1 / min, for Q2) objective value improve? |
| **Population diversity** | How many *distinct* chromosomes remain? (4 = fully diverse, 1 = fully collapsed) |
| **Fitness variance** | How spread out are the fitness values across the population? (0 = everyone identical) |

If diversity has dropped sharply and/or variance ≈ 0 **and** the best value stopped improving, the
population has (prematurely) converged. If the best value is still improving and diversity remains
high, the GA has **not yet converged** and needs more generations.

---
## Part A — Q1: Maximize $f(x) = x^2$, $x \in [0, 31]$ (5-bit binary encoding)


### A.1 Setup — encode / decode / fitness

In [1]:

import pandas as pd
import matplotlib.pyplot as plt

CHROM_LENGTH = 5   # 5 bits -> x in [0, 31]

def decode(chromosome: str) -> int:
    return int(chromosome, 2)

def fitness_q1(x: int) -> int:
    '''f(x) = x^2  (to MAXIMIZE)'''
    return x ** 2

def crossover(parent1: str, parent2: str, point: int):
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

def mutate_at(chromosome: str, position: int) -> str:
    bits = list(chromosome)
    bits[position] = '1' if bits[position] == '0' else '0'
    return ''.join(bits)

def build_generation_table(population, fitness_fn, mode='max'):
    '''Fitness / probability / expected count / actual count table for one generation.'''
    d = pd.DataFrame({'Chromosome': population})
    d['x'] = d['Chromosome'].apply(decode)
    d['f(x)'] = d['x'].apply(fitness_fn)

    if mode == 'max':
        d['Probability'] = (d['f(x)'] / d['f(x)'].sum()).round(3)
        d['Expected count'] = (d['f(x)'] / d['f(x)'].mean()).round(3)
    else:  # 'min' -> invert so higher selection-fitness = smaller f(x)
        worst = d['f(x)'].max()
        d['Selection fitness'] = worst - d['f(x)'] + 1e-6
        d['Probability'] = (d['Selection fitness'] / d['Selection fitness'].sum()).round(3)
        d['Expected count'] = (d['Selection fitness'] / d['Selection fitness'].mean()).round(3)

    # guaranteed (floor) + largest-remainder round-off, exactly as in the worked example
    d['Guaranteed'] = d['Expected count'].apply(lambda v: int(v))
    remaining = len(d) - d['Guaranteed'].sum()
    remainder = d['Expected count'] - d['Guaranteed']
    bonus = pd.Series(0, index=d.index)
    top_idx = remainder.sort_values(ascending=False).index[:remaining]
    bonus.loc[top_idx] = 1
    d['Actual count'] = d['Guaranteed'] + bonus

    return d

def mating_pool_from(table):
    pool = []
    for _, row in table.iterrows():
        pool.extend([row['Chromosome']] * int(row['Actual count']))
    return pool


### A.2 Population 1 (Initial) — fitness, probability, expected/actual count

In [2]:

population_1_q1 = ['11011', '10001', '01111', '10111']

table1_q1 = build_generation_table(population_1_q1, fitness_q1, mode='max')
print(f"Sum f(x) = {table1_q1['f(x)'].sum()}   Average f(x) = {table1_q1['f(x)'].mean():.1f}   Max f(x) = {table1_q1['f(x)'].max()}")
table1_q1


Sum f(x) = 1772   Average f(x) = 443.0   Max f(x) = 729


,Chromosome,x,f(x),Probability,Expected count,Guaranteed,Actual count
0,11011,27,729,0.411,1.646,1,2
1,10001,17,289,0.163,0.652,0,1
2,01111,15,225,0.127,0.508,0,0
3,10111,23,529,0.299,1.194,1,1


### A.3 Mating pool (from Population 1)

In [3]:

pool_q1 = mating_pool_from(table1_q1)
print("Mating pool:", pool_q1)


Mating pool: ['11011', '11011', '10001', '10111']



### A.4 Crossover → A.5 Mutation → **Population 2**

Same pattern as the worked example: pair consecutively `(1↔2), (3↔4)`, single-point crossover
(cut after bit 3), then bit-flip mutation on the two offspring from the identical-parent pair
(the pair that crossover alone cannot change).


In [4]:

pairs_q1 = [(pool_q1[0], pool_q1[1]), (pool_q1[2], pool_q1[3])]
cut = 3

offspring_q1 = []
for p1, p2 in pairs_q1:
    c1, c2 = crossover(p1, p2, cut)
    offspring_q1.extend([c1, c2])
print("After crossover:", offspring_q1)

# Mutation (matches the worked example's next generation)
offspring_q1[0] = mutate_at(offspring_q1[0], 2)   # 11011 -> 11111
offspring_q1[1] = mutate_at(offspring_q1[1], 3)   # 11011 -> 11001

population_2_q1 = offspring_q1
print("Population 2:", population_2_q1)


After crossover: ['11011', '11011', '10011', '10101']
Population 2: ['11111', '11001', '10011', '10101']


### A.6 Population 2 — fitness, probability, expected/actual count

In [5]:

table2_q1 = build_generation_table(population_2_q1, fitness_q1, mode='max')
print(f"Sum f(x) = {table2_q1['f(x)'].sum()}   Average f(x) = {table2_q1['f(x)'].mean():.1f}   Max f(x) = {table2_q1['f(x)'].max()}")
table2_q1


Sum f(x) = 2388   Average f(x) = 597.0   Max f(x) = 961


,Chromosome,x,f(x),Probability,Expected count,Guaranteed,Actual count
0,11111,31,961,0.402,1.610,1,2
1,11001,25,625,0.262,1.047,1,1
2,10011,19,361,0.151,0.605,0,0
3,10101,21,441,0.185,0.739,0,1


### A.7 Convergence check — Population 1 vs Population 2 (Q1)

In [6]:

def convergence_report(table1, table2, mode, label, true_optimum=None):
    best1 = table1['f(x)'].max() if mode == 'max' else table1['f(x)'].min()
    best2 = table2['f(x)'].max() if mode == 'max' else table2['f(x)'].min()
    div1, div2 = table1['Chromosome'].nunique(), table2['Chromosome'].nunique()
    var1, var2 = table1['f(x)'].var(ddof=0), table2['f(x)'].var(ddof=0)

    improved = (best2 > best1) if mode == 'max' else (best2 < best1)

    print(f"--- Convergence Report: {label} ---")
    print(f"{'Metric':<22}{'Population 1':>15}{'Population 2':>15}")
    print(f"{'Best f(x)':<22}{best1:>15.4f}{best2:>15.4f}")
    print(f"{'Diversity (unique)':<22}{div1:>15}{div2:>15}")
    print(f"{'Fitness variance':<22}{var1:>15.4f}{var2:>15.4f}")
    if true_optimum is not None:
        print(f"{'True optimum f(x)':<22}{'':>15}{true_optimum:>15.4f}")

    print()
    if not improved and (div2 <= 2 or var2 < 1e-3):
        verdict = "CONVERGED (or converging): best value stopped improving and diversity/variance has collapsed."
    elif improved and div2 == len(table2):
        verdict = "NOT YET CONVERGED: best value is still improving and full diversity remains -- continue running more generations."
    else:
        verdict = "PARTIALLY CONVERGED: some signals point to convergence, others don't -- run a few more generations to be sure."
    print("Verdict:", verdict)
    return verdict

verdict_q1 = convergence_report(table1_q1, table2_q1, mode='max', label='Q1 (maximize x^2)', true_optimum=961)


--- Convergence Report: Q1 (maximize x^2) ---
Metric                   Population 1   Population 2
Best f(x)                    729.0000       961.0000
Diversity (unique)                  4              4
Fitness variance           40108.0000     53328.0000
True optimum f(x)                           961.0000

Verdict: NOT YET CONVERGED: best value is still improving and full diversity remains -- continue running more generations.



**Interpretation (Q1):** the best fitness jumped from **729 → 961** and every chromosome in
Population 2 is still distinct (diversity = 4/4). Both signals say the search is still actively
improving, so with only two generations completed **Q1 has not yet converged** — more generations
are needed (in the earlier multi-generation run, this same setup converged to the true global
optimum $x=31,\ f(x)=961$ within about 3 generations).



---
## Part B — Q2: Minimize $f(x) = -x^2 + 2x$, $x \in [0, 2]$ (real-valued decoding)

Same pattern, adapted for a real-valued decode and a **minimization** objective (selection fitness
is rescaled as *worst − f(x)*, and roulette-wheel selection uses the given random numbers
$r_1=0.4,\ r_2=0.15,\ r_3=0.7,\ r_4=0.9$ directly, exactly as worked out by hand).


### B.1 Setup — real-valued decoding

In [7]:

def decode_real(chromosome: str, x_min: float, x_max: float, length: int = CHROM_LENGTH) -> float:
    d = int(chromosome, 2)
    return x_min + (x_max - x_min) / (2 ** length - 1) * d

def fitness_q2(x: float) -> float:
    '''f(x) = -x^2 + 2x  (to MINIMIZE)'''
    return -x**2 + 2*x

X_MIN, X_MAX = 0, 2


### B.2 Population 1 (Initial) — decode, fitness, selection probability

In [8]:

population_1_q2 = ['11010', '00111', '10110', '00101']
random_numbers_q2 = [0.4, 0.15, 0.7, 0.9]   # r1..r4, as given

table1_q2 = pd.DataFrame({'Chromosome': population_1_q2, 'r': random_numbers_q2})
table1_q2['x'] = table1_q2['Chromosome'].apply(lambda c: decode_real(c, X_MIN, X_MAX))
table1_q2['f(x)'] = table1_q2['x'].apply(fitness_q2)

# Selection fitness (invert for minimization) and cumulative roulette-wheel ranges
worst = table1_q2['f(x)'].max()
table1_q2['Selection fitness'] = worst - table1_q2['f(x)'] + 1e-9
table1_q2['Probability'] = table1_q2['Selection fitness'] / table1_q2['Selection fitness'].sum()
table1_q2['Cumulative'] = table1_q2['Probability'].cumsum()

table1_q2.round(4)


,Chromosome,r,x,f(x),Selection fitness,Probability,Cumulative
0,11010,0.40,1.6774,0.5411,0.2830,0.4096,0.4096
1,00111,0.15,0.4516,0.6993,0.1249,0.1807,0.5904
2,10110,0.70,1.4194,0.8241,0.0000,0.0000,0.5904
3,00101,0.90,0.3226,0.5411,0.2830,0.4096,1.0000


### B.3 Roulette-wheel selection using r1..r4 → mating pool

In [9]:

def roulette_select(table, r):
    cum = 0.0
    for _, row in table.iterrows():
        cum = row['Cumulative']
        if r <= cum:
            return row['Chromosome']
    return table.iloc[-1]['Chromosome']   # safety net for r == 1.0 edge case

pool_q2 = [roulette_select(table1_q2, r) for r in random_numbers_q2]
print("Mating pool:", pool_q2)


Mating pool: ['11010', '11010', '00101', '00101']



### B.4 Crossover → B.5 Mutation → **Population 2**

The mating pool pairs up as two sets of *identical twins* (`11010`×`11010` and `00101`×`00101`),
so single-point crossover — regardless of the exact cut — leaves both pairs unchanged. No mutation
rule was specified for Q2 in the notes, so none is applied here.


In [10]:

pairs_q2 = [(pool_q2[0], pool_q2[1]), (pool_q2[2], pool_q2[3])]
cut_q2 = 3   # any cut point gives the same result for identical parents

offspring_q2 = []
for p1, p2 in pairs_q2:
    c1, c2 = crossover(p1, p2, cut_q2)
    offspring_q2.extend([c1, c2])

population_2_q2 = offspring_q2
print("Population 2:", population_2_q2)


Population 2: ['11010', '11010', '00101', '00101']


### B.6 Population 2 — decode, fitness

In [11]:

table2_q2 = pd.DataFrame({'Chromosome': population_2_q2})
table2_q2['x'] = table2_q2['Chromosome'].apply(lambda c: decode_real(c, X_MIN, X_MAX))
table2_q2['f(x)'] = table2_q2['x'].apply(fitness_q2)
table2_q2.round(4)


,Chromosome,x,f(x)
0,11010,1.6774,0.5411
1,11010,1.6774,0.5411
2,00101,0.3226,0.5411
3,00101,0.3226,0.5411


### B.7 Convergence check — Population 1 vs Population 2 (Q2)

In [12]:

verdict_q2 = convergence_report(table1_q2, table2_q2, mode='min', label='Q2 (minimize -x^2+2x)', true_optimum=0.0)


--- Convergence Report: Q2 (minimize -x^2+2x) ---
Metric                   Population 1   Population 2
Best f(x)                      0.5411         0.5411
Diversity (unique)                  4              2
Fitness variance               0.0141         0.0000
True optimum f(x)                             0.0000

Verdict: CONVERGED (or converging): best value stopped improving and diversity/variance has collapsed.



**Interpretation (Q2):** diversity collapsed from 4 unique chromosomes down to just **2**
(`11010` and `00101`, each doubled), and the fitness variance across the population dropped to
**≈0** — every individual in Population 2 has the same f(x). The best value itself did **not**
improve between generations (it was already the minimum available in Population 1). All three
signals agree: **Q2 has converged after just two generations** — but only to a *local* plateau
($f(x)\approx0.541$ at $x\approx1.677$ and $x\approx0.323$), not the true global minimum
($f(x)=0$ at the boundaries $x=0$ or $x=2$). This is a textbook case of **premature convergence**:
once the mating pool contains only duplicate pairs, crossover cannot introduce anything new, and
with no mutation applied, the population is stuck. Adding a small mutation probability would let it
keep drifting toward a true boundary.


---
## Summary — Q1 vs Q2 convergence behaviour

In [13]:

summary = pd.DataFrame([
    {
        'Problem': 'Q1: maximize x^2',
        'Best (Pop 1)': table1_q1['f(x)'].max(),
        'Best (Pop 2)': table2_q1['f(x)'].max(),
        'Diversity (Pop 1)': table1_q1['Chromosome'].nunique(),
        'Diversity (Pop 2)': table2_q1['Chromosome'].nunique(),
        'Converged after 2 gens?': 'No -- still improving'
    },
    {
        'Problem': 'Q2: minimize -x^2+2x',
        'Best (Pop 1)': round(table1_q2['f(x)'].min(), 4),
        'Best (Pop 2)': round(table2_q2['f(x)'].min(), 4),
        'Diversity (Pop 1)': table1_q2['Chromosome'].nunique(),
        'Diversity (Pop 2)': table2_q2['Chromosome'].nunique(),
        'Converged after 2 gens?': 'Yes (prematurely, to a local plateau)'
    }
])
summary


,Problem,Best (Pop 1),Best (Pop 2),Diversity (Pop 1),Diversity (Pop 2),Converged after 2 gens?
0,Q1: maximize x^2,729.0000,961.0000,4,4,No -- still improving
1,Q2: minimize -x^2+2x,0.5411,0.5411,4,2,"Yes (prematurely, to a local plateau)"



### Why the two problems behave so differently after just two generations

* **Q1** started with four *distinct-fitness* strings spread across a wide range (225–729), so
  selection had clear winners to favour but the population itself stayed diverse — crossover between
  non-identical parents kept generating new combinations, and fitness kept climbing.
* **Q2** started with two pairs of near-tied fitness values, and the roulette wheel (driven by the
  given random numbers) happened to pick the *same* two chromosomes twice each. Once the mating pool
  is made of identical twins, crossover is powerless to add diversity, so the population converges
  immediately — but to whatever those two chromosomes happened to represent, not necessarily the true
  optimum.

This illustrates a core practical lesson in GAs: **convergence speed and convergence quality are not
the same thing** — a population can converge very quickly (Q2) while still being far from the true
optimum, whereas a slower-converging population (Q1) may be steadily working its way toward it.
